In [22]:
import numpy as np
import arlpy.signal as asig
from arlpy.plot import *
from unetpy import *

### Connect to the modem

In [23]:
modem = UnetGateway('localhost')

In [24]:
phy = modem.agent('phy')
modem.subscribe(phy)

### Set passband block count & signal power level

In [25]:
fs = 48000
phy.pbsblk = 48000
#phy.signalPowerLevel = -10

In [26]:
def flush_modem():
    while modem.receive(timeout=1000):
        pass

### Generater the signal to be transmitted

In [6]:
tx = asig.cw(5000, 50e-3, fs*2).tolist()

### Start baseband signal reception and transmit the signal

In [28]:
flush_modem()
phy.npulses = 5
phy.pulsedelay = 100
phy.pbscnt = 1
phy << org_arl_unet_bb.TxBasebandSignalReq(signal=tx, fc=0)
rx = modem.receive(org_arl_unet_bb.RxBasebandSignalNtf, 5000)
txntf = modem.receive(timeout=5000)
txTime = txntf.txTime
rxTime = rx.rxTime

### Plot the recorded signal & PSD

In [29]:
plot(rx.signal, fs=rx.fs, maxpts=48000)

In [30]:
psd(rx.signal, fs=fs, nfft=512, noverlap=None, window='hanning', color=None, style='solid', thickness=1, marker=None, filled=False, size=6, title=None, xlabel='Frequency (Hz)', ylabel='Power spectral density (dB/Hz)', xlim=None, ylim=None, width=None, height=None, hold=False, interactive=None)

In [18]:
env = asig.envelope(rx.signal)
threshold = 0.05
edge = np.flatnonzero((env[:-1] < threshold) & (env[1:] > threshold))+1

In [19]:
plot(env, fs=fs, maxpts = 48000, hold=True)
txStart=(txTime-rxTime)/1e6
rxStart=(edge[0]/fs)
vlines(txStart, hold=True, color='orange')
vlines(rxStart, color='purple')

In [38]:
print ('Tx Start = ', txStart, 's')
print ('Rx Start = ', rxStart, 's')
travelTime = rxStart-txStart
print ('Round trip time = ', travelTime, 's')
soundSpeed = 60
print ('Range = ', soundSpeed * travelTime, 'm')

Tx Start =  0.021333 s
Rx Start =  0.23789583333333333 s
Round trip time =  0.21656283333333334 s
Range =  12.993770000000001 m


In [39]:
modem.shutdown()